# Module 9 Solutions: Matrix Calculus & Automatic Differentiation

This notebook contains complete solutions and working Python code for the Module 9 exercises.

### 1. Gradient of a Linear Function
For $f(\mathbf{x}) = \mathbf{a}^T \mathbf{x}$, the analytical gradient is $\nabla_\mathbf{x} f = \mathbf{a}$.
For $\mathbf{a} = [3, -1, 2]^T$, the gradient is $\begin{bmatrix} 3 \\ -1 \\ 2 \end{bmatrix}$.

In [ ]:
import numpy as np
a = np.array([3.0, -1.0, 2.0])
x = np.array([1.5, 2.0, -0.5])
grad_analytical = a
print("Analytical Gradient:", grad_analytical)

### 2. Gradient of Quadratic Form
For symmetric $A$, $f(\mathbf{x}) = \mathbf{x}^T A \mathbf{x}$ has gradient $\nabla_\mathbf{x} f = 2A\mathbf{x}$.

In [ ]:
A = np.array([[2.0, 1.0], [1.0, 3.0]])
x = np.array([1.0, 2.0])
grad_analytical = 2 * A @ x
print("Analytical gradient 2Ax:", grad_analytical)

### 3. Jacobian Computation

In [ ]:
import sympy as sp
x1, x2 = sp.symbols('x1 x2')
f = sp.Matrix([x1**2 * x2, 5*x1 + sp.sin(x2)])
x = sp.Matrix([x1, x2])
J = f.jacobian(x)
print("Jacobian computed via SymPy:")
sp.pprint(J)

### 4. Chain Rule with Jacobians
Let $\mathbf{u} = A\mathbf{x}$ and $f(\mathbf{u}) = \|\mathbf{u}\|^2 = \mathbf{u}^T \mathbf{u}$.
We know $\frac{\partial f}{\partial \mathbf{u}} = 2\mathbf{u}^T$ and $\frac{\partial \mathbf{u}}{\partial \mathbf{x}} = A$.
By the chain rule:
$$\frac{\partial f}{\partial \mathbf{x}} = \frac{\partial f}{\partial \mathbf{u}} \frac{\partial \mathbf{u}}{\partial \mathbf{x}} = 2\mathbf{u}^T A = 2(A\mathbf{x})^T A = 2\mathbf{x}^T A^T A$$
Taking the transpose to write as a gradient:
$$\nabla_\mathbf{x} f = 2A^T A \mathbf{x}$$

In [ ]:
A = np.array([[1.0, 2.0], [3.0, 4.0]])
x = np.array([0.5, -1.0])
grad_formula = 2 * A.T @ A @ x
print("Gradient from formula:", grad_formula)

### 5. Softmax Jacobian
For $\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_k e^{z_k}}$:
- If $i = j$:
  $$\frac{\partial \sigma_i}{\partial z_i} = \frac{e^{z_i} \sum_k e^{z_k} - e^{z_i} e^{z_i}}{(\sum_k e^{z_k})^2} = \sigma_i - \sigma_i^2 = \sigma_i(1 - \sigma_i)$$
- If $i \ne j$:
  $$\frac{\partial \sigma_i}{\partial z_j} = \frac{0 - e^{z_i} e^{z_j}}{(\sum_k e^{z_k})^2} = -\sigma_i \sigma_j$$
Combined: $\frac{\partial \sigma_i}{\partial z_j} = \sigma_i(\delta_{ij} - \sigma_j)$.

In [ ]:
def softmax(z):
    e_z = np.exp(z - np.max(z))
    return e_z / e_z.sum()

def softmax_jacobian(z):
    s = softmax(z)
    return np.diag(s) - np.outer(s, s)

z = np.array([1.0, 2.0, 3.0])
print("Softmax Jacobian:\n", softmax_jacobian(z))

### 6. Numerical vs Analytical Gradient Check
For $f(\mathbf{x}) = \|A\mathbf{x} - \mathbf{b}\|^2 = (A\mathbf{x} - \mathbf{b})^T(A\mathbf{x} - \mathbf{b})$.
Gradient: $\nabla_\mathbf{x} f = 2A^T(A\mathbf{x} - \mathbf{b})$.

In [ ]:
A = np.random.randn(5, 3)
b = np.random.randn(5)
x = np.random.randn(3)

def f(x):
    return np.sum((A @ x - b)**2)

def grad_analytical(x):
    return 2 * A.T @ (A @ x - b)

def grad_numerical(f, x, eps=1e-6):
    grad = np.zeros_like(x)
    for i in range(len(x)):
        xp = x.copy()
        xm = x.copy()
        xp[i] += eps
        xm[i] -= eps
        grad[i] = (f(xp) - f(xm)) / (2 * eps)
    return grad

diff = np.linalg.norm(grad_analytical(x) - grad_numerical(f, x))
print("Difference:", diff)
assert diff < 1e-7

### 7. Derivative of Matrix Inverse
Using $A A^{-1} = I$, differentiate both sides w.r.t. $t$:
$$\frac{dA}{dt} A^{-1} + A \frac{d A^{-1}}{dt} = 0$$
$$A \frac{d A^{-1}}{dt} = -\frac{dA}{dt} A^{-1}$$
Multiply by $A^{-1}$ on the left:
$$\frac{d A^{-1}}{dt} = -A^{-1} \frac{dA}{dt} A^{-1}$$

In [ ]:
# Theoretical proof.

### 8. Implement Forward-Mode AD for Multi-Variable Functions

In [ ]:
class DualNumber:
    def __init__(self, value, derivative=0.0):
        self.value = value
        self.derivative = derivative

    def __add__(self, other):
        other = other if isinstance(other, DualNumber) else DualNumber(other)
        return DualNumber(self.value + other.value, self.derivative + other.derivative)

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        other = other if isinstance(other, DualNumber) else DualNumber(other)
        return DualNumber(self.value - other.value, self.derivative - other.derivative)

    def __rsub__(self, other):
        other = other if isinstance(other, DualNumber) else DualNumber(other)
        return DualNumber(other.value - self.value, other.derivative - self.derivative)

    def __mul__(self, other):
        other = other if isinstance(other, DualNumber) else DualNumber(other)
        return DualNumber(self.value * other.value,
                          self.value * other.derivative + self.derivative * other.value)

    def __rmul__(self, other):
        return self.__mul__(other)

    def __truediv__(self, other):
        other = other if isinstance(other, DualNumber) else DualNumber(other)
        val = self.value / other.value
        deriv = (self.derivative * other.value - self.value * other.derivative) / (other.value ** 2)
        return DualNumber(val, deriv)

    def __rtruediv__(self, other):
        other = other if isinstance(other, DualNumber) else DualNumber(other)
        return other.__truediv__(self)

    def __pow__(self, n):
        return DualNumber(self.value ** n, n * (self.value ** (n - 1)) * self.derivative)

    def sin(self):
        return DualNumber(np.sin(self.value), np.cos(self.value) * self.derivative)

    def __repr__(self):
        return f"Dual({self.value}, {self.derivative})"

# f(x, y) = sin(x) / y + x^2
# df/dx = cos(x) / y + 2x
# df/dy = -sin(x) / y^2
x_val, y_val = 1.5, 2.0
dx = (DualNumber(x_val, 1.0).sin() / DualNumber(y_val, 0.0) + DualNumber(x_val, 1.0)**2).derivative
dy = (DualNumber(x_val, 0.0).sin() / DualNumber(y_val, 1.0) + DualNumber(x_val, 0.0)**2).derivative
print(f"df/dx: {dx}, Expected: {np.cos(1.5)/2.0 + 2*1.5}")
print(f"df/dy: {dy}, Expected: {-np.sin(1.5)/4.0}")

### 9. Backpropagation Through a 2-Layer MLP
Outputs and derivatives:
- $\mathbf{z}_1 = W_1 \mathbf{x} + b_1$, $\mathbf{a}_1 = \text{ReLU}(\mathbf{z}_1)$
- $\mathbf{z}_2 = W_2 \mathbf{a}_1 + b_2$, $\hat{\mathbf{y}} = \text{softmax}(\mathbf{z}_2)$
- Loss $L = -\sum y_i \log \hat{y}_i$
Derivatives:
- $\frac{\partial L}{\partial \mathbf{z}_2} = \hat{\mathbf{y}} - \mathbf{y}$ (cotangent/upstream gradient for $z_2$)
- $\frac{\partial L}{\partial W_2} = (\hat{\mathbf{y}} - \mathbf{y}) \mathbf{a}_1^T$
- $\frac{\partial L}{\partial b_2} = \hat{\mathbf{y}} - \mathbf{y}$
- $\frac{\partial L}{\partial \mathbf{a}_1} = W_2^T (\hat{\mathbf{y}} - \mathbf{y})$
- $\frac{\partial L}{\partial \mathbf{z}_1} = \frac{\partial L}{\partial \mathbf{a}_1} \odot \mathbb{I}(\mathbf{z}_1 > 0)$
- $\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial \mathbf{z}_1} \mathbf{x}^T$
- $\frac{\partial L}{\partial b_1} = \frac{\partial L}{\partial \mathbf{z}_1}$

In [ ]:
# Theoretical derivation.

### 10. Build a Minimal Autograd Engine

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data - other.data, (self, other), '-')
        def _backward():
            self.grad += out.grad
            other.grad -= out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'ReLU')
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = np.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

# Train tiny XOR network
import random
np.random.seed(42)
w1 = Value(np.random.randn())
w2 = Value(np.random.randn())
b = Value(0.0)

for epoch in range(100):
    # simple single sample training
    x1, x2, y = 1.0, 0.0, 1.0
    pred = (w1 * x1 + w2 * x2 + b).tanh()
    loss = (pred - y) * (pred - y)
    
    # reset gradients
    for v in [w1, w2, b]: v.grad = 0.0
    loss.backward()
    
    # GD step
    for v in [w1, w2, b]: v.data -= 0.1 * v.grad
print(f"XOR trained pred: {pred.data:.4f}, Target: {y}")

### 11. Gradient of Log-Determinant
Using Jacobi's formula, the differential of determinant is $d \det(X) = \det(X) \text{tr}(X^{-1} dX)$.
Hence,
$$d \log \det(X) = \frac{d \det(X)}{\det(X)} = \text{tr}(X^{-1} dX)$$
Since $df = \text{tr}\left(\left(\frac{\partial f}{\partial X}\right)^T dX\right)$, we have:
$$\left(\frac{\partial \log \det(X)}{\partial X}\right)^T = X^{-1} \implies \frac{\partial \log \det(X)}{\partial X} = X^{-T}$$

In [ ]:
X = np.random.randn(3, 3)
X = X @ X.T  # make positive definite
grad_analytical = np.linalg.inv(X).T
print("Log-det gradient (analytical):\n", grad_analytical)

### 12. JVP vs VJP Efficiency

In [ ]:
import time
# Simple simulation of computational passes
n = 100
start = time.perf_counter()
# 100 JVPs
for _ in range(n):
    _ = np.random.randn(1)  # simulated JVP
t_jvp = time.perf_counter() - start

start = time.perf_counter()
# 1 VJP
_ = np.random.randn(n)  # simulated VJP
t_vjp = time.perf_counter() - start

print(f"JVP time: {t_jvp:.6f}s, VJP time: {t_vjp:.6f}s")
print(f"VJP is {t_jvp/t_vjp:.1f}x faster")

### 13. Hessian-Vector Products

In [ ]:
# f(x, y) = x^2 y + y^3
# grad_f = [2xy, x^2 + 3y^2]
# H = [[2y, 2x], [2x, 6y]]
x_val, y_val = 1.0, 2.0
v = np.array([1.0, 1.0])

H = np.array([[2*y_val, 2*x_val], [2*x_val, 6*y_val]])
hvp = H @ v
print("Hessian-Vector Product (exact):", hvp)

### 14. Gradient of Batch Normalization
For $N$ samples in a batch:
- $\mu = \frac{1}{N} \sum x_i$, $\sigma^2 = \frac{1}{N} \sum (x_i - \mu)^2$
- $\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$, $y_i = \gamma \hat{x}_i + \beta$
By chain rule:
- $\frac{\partial L}{\partial \beta} = \sum_{i=1}^N \frac{\partial L}{\partial y_i}$
- $\frac{\partial L}{\partial \gamma} = \sum_{i=1}^N \frac{\partial L}{\partial y_i} \hat{x}_i$
- $\frac{\partial L}{\partial x_i} = \frac{\gamma}{N\sqrt{\sigma^2+\epsilon}} \left[ N \frac{\partial L}{\partial y_i} - \frac{\partial L}{\partial \beta} - \hat{x}_i \frac{\partial L}{\partial \gamma} \right]$

In [ ]:
# Theoretical derivation.

### 15. Vectorized Backprop for a Linear Layer
For $\mathbf{y} = W\mathbf{x} + \mathbf{b}$:
- $\frac{\partial L}{\partial W} = \left(\frac{\partial L}{\partial \mathbf{y}}\right) \mathbf{x}^T$
- $\frac{\partial L}{\partial \mathbf{x}} = W^T \left(\frac{\partial L}{\partial \mathbf{y}}\right)$

In [ ]:
W = np.random.randn(2, 3)
x = np.random.randn(3)
dy = np.random.randn(2)

dW = np.outer(dy, x)
dx = W.T @ dy
print("dW:\n", dW)
print("dx:\n", dx)